# Endoscopy SRGAN Training — Clean Setup

Self-healing notebook: safe to re-run from the top in any session (fresh or resumed).
Each cell checks what already exists before doing expensive work again.

**Order matters the first time; after that, just re-run top to bottom.**

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Kaggle authentication

**Only needed the very first time ever** (once a Drive backup of the dataset exists, cell 5 skips Kaggle entirely on every future run). Get a fresh token from kaggle.com → Settings → API → Create New Token, paste it below.

In [ ]:
import os

KAGGLE_TOKEN = 'PASTE_YOUR_TOKEN_HERE'  # e.g. KGAT_xxxxxxxx... -- skip if a Drive backup already exists (see cell 5)

if KAGGLE_TOKEN != 'PASTE_YOUR_TOKEN_HERE':
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/access_token', 'w') as f:
        f.write(KAGGLE_TOKEN)
    os.system('chmod 600 /root/.kaggle/access_token')
    print('Kaggle token saved.')
else:
    print('Skipped -- no token provided. Fine if a Drive backup already exists (cell 5 will use it).')

## 3. Install dependencies

In [ ]:
!pip install -q kaggle opencv-python-headless torch torchvision scikit-image lpips

## 4. Get the dataset — local disk first, Drive as backup only

Downloads/training READ from `/content/data` (fast local disk), never directly from
Drive -- this is what was making training slow. Drive is used only as a persistent
backup so you don't need to re-download from Kaggle every session.

Handles all three cases automatically:
- Already downloaded this session -> does nothing
- Drive backup exists (from a previous session) -> fast local restore, no Kaggle needed
- Neither exists (first time ever) -> downloads from Kaggle straight to local disk
  (avoids the `[Errno 107] Transport endpoint is not connected` failure that happens
  downloading large files directly onto the Drive FUSE mount), then backs up to Drive

In [ ]:
import os

LOCAL_DATA = '/content/data'
DRIVE_BACKUP = '/content/drive/MyDrive/endoscopy_srgan/data_backup'
DATASETS = ['kvasir', 'cvc_clinicdb', 'etis_larib']

def _has_all(root):
    return all(os.path.isdir(f'{root}/{d}') for d in DATASETS)

if _has_all(LOCAL_DATA):
    print('Local data already present this session -- skipping.')
elif _has_all(DRIVE_BACKUP):
    print('Restoring from Drive backup (fast local copy)...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    for d in DATASETS:
        os.system(f'cp -r {DRIVE_BACKUP}/{d} {LOCAL_DATA}/')
    print('Restored from Drive backup.')
else:
    print('No local data or Drive backup found -- downloading fresh from Kaggle to local disk...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    os.system(f'kaggle datasets download -d meetnagadia/kvasir-dataset -p {LOCAL_DATA}/kvasir --unzip')
    os.system(f'kaggle datasets download -d balraj98/cvcclinicdb -p {LOCAL_DATA}/cvc_clinicdb --unzip')
    os.system(f'kaggle datasets download -d nguyenvoquocduong/etis-laribpolypdb -p {LOCAL_DATA}/etis_larib --unzip')
    print('Downloaded. Backing up to Drive for future sessions...')
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    for d in DATASETS:
        os.system(f'cp -r {LOCAL_DATA}/{d} {DRIVE_BACKUP}/')
    print('Backed up to Drive -- future sessions will restore from here, no Kaggle needed.')

## 5. Verify the dataset

In [ ]:
for d in os.listdir(LOCAL_DATA):
    path = os.path.join(LOCAL_DATA, d)
    if not os.path.isdir(path):
        continue
    print(d, '->', len(os.listdir(path)), 'items:', os.listdir(path)[:5])

## 6. Build the manifest (local paths, so training reads local disk)

In [ ]:
import glob, random, json

paths = []
paths += glob.glob(f'{LOCAL_DATA}/kvasir/kvasir-dataset/*/*.jpg')
paths += glob.glob(f'{LOCAL_DATA}/cvc_clinicdb/PNG/Original/*.png')
paths += glob.glob(f'{LOCAL_DATA}/etis_larib/images/*.png')
print('total images found:', len(paths))

random.seed(42)  # fixed seed -- same train/val split every time
random.shuffle(paths)
n_val = int(0.1 * len(paths))
manifest = {'train': paths[n_val:], 'val': paths[:n_val]}

MANIFEST_PATH = f'{LOCAL_DATA}/manifest.json'
with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f)
print('train:', len(manifest['train']), '| val:', len(manifest['val']))
print('saved to', MANIFEST_PATH)

## 7. Clone or update the repo

In [ ]:
import os

if os.path.exists('/content/repo/.git'):
    %cd /content/repo
    !git pull
else:
    !git clone https://github.com/knah1d/unsharp-image_processing.git /content/repo
    %cd /content/repo

## 8. Smoke test (recommended before a long run)

Cheap sanity check -- catches bugs in ~1 minute before committing GPU time to a full run.
Uses a throwaway checkpoint dir so it never touches your real checkpoints.

In [ ]:
!python train_srgan.py \
    --manifest /content/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_smoketest \
    --epochs 3 --pretrain_epochs 1 --batch_size 8

## 9. Real training run

Checkpoints go to Drive (persistent across disconnects); data reads come from local
disk (fast). Auto-resumes if `srgan_last.pth` already exists in `--ckpt_dir` --
just rerun this exact cell after a disconnect, no need to redo anything above
except cells 1, 3 (deps), 4 (fast local restore), 6 (manifest), 7 (repo) if it's
a fresh runtime.

In [ ]:
!python train_srgan.py \
    --manifest /content/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints \
    --epochs 80 --fresh_schedule

## 10. (Optional) Transfer-learning alternative

Fine-tunes a pretrained Real-ESRGAN generator instead of training from scratch --
a separate, opt-in path, not a replacement for the paper-faithful model above.

In [ ]:
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth

!python train_rrdb.py \
    --manifest /content/data/manifest.json \
    --ckpt_dir /content/drive/MyDrive/endoscopy_srgan/checkpoints_rrdb \
    --pretrained RealESRGAN_x2plus.pth \
    --epochs 30 --pretrain_epochs 2 --batch_size 8